# Argo Profile CSV Structure

This notebook shows how an Argo profile CSV becomes a clean tabular DataFrame.

## File Structure

Each profile CSV has two parts:

- A metadata comment block at the top (lines starting with `#`).
- A normal comma-separated table below with a header row and measurement rows.

Typical metadata in the comment block includes:

- format and creation info
- platform number and cycle number
- profile date and coordinates
- variable descriptions and units

Typical table columns include:

- pressure
- temperature
- salinity
- pressure_error
- temperature_error
- salinity_error

## How It Becomes Clean Tabular Data

Use pandas with `comment='#'` so comment metadata is skipped while the real header and rows are parsed:

```python
df = pd.read_csv(csv_path, comment='#')
```

If you removed a large folder from the repo, set a placeholder path in your own run, for example:

```python
csv_path = Path("<YOUR_BIG_FOLDER>") / "<platform_or_subfolder>" / "<file>.csv"
```

Then parse metadata from comment lines and append it as `meta_*` columns so every row keeps profile context.

## Notes

- Parsing errors usually happen when metadata comment lines are read as data.
- With `comment='#'`, pandas reads only the clean table section.

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re


profiles_root = Path('data/raw/profiles')
sample_files = sorted(profiles_root.rglob('*.csv'))
if not sample_files:
    raise FileNotFoundError(f'No profile CSV files found under {profiles_root}')

csv_path = sample_files[0]
argo_metadata = {}
with csv_path.open('r', encoding='utf-8', errors='replace') as csv_file:
    for line in csv_file:
        if not line.startswith('#'):
            break
        comment = line[1:].strip()
        if '=' in comment:
            key, value = comment.split('=', 1)
            argo_metadata[key.strip()] = value.strip()
        else:
            parts = comment.split(None, 1)
            if len(parts) == 2:
                argo_metadata[parts[0].strip()] = parts[1].strip()
            else:
                argo_metadata[comment] = None

single_argo = pd.read_csv(csv_path, comment='#')

# Add one constant metadata column per key for every row in the profile table.
for key, value in argo_metadata.items():
    safe_key = re.sub(r'[^0-9a-zA-Z_]+', '_', key.strip().lower()).strip('_')
    meta_col = f"meta_{safe_key}" if safe_key else "meta_unknown"
    single_argo[meta_col] = value

single_argo.attrs['metadata'] = argo_metadata
print('Loaded sample file:', csv_path)
single_argo.head()

In [ ]:
from pathlib import Path
import pandas as pd
import re


def _parse_argo_metadata(csv_file):
    metadata = {}
    with Path(csv_file).open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.startswith("#"):
                break
            comment = line[1:].strip()
            if not comment:
                continue
            if "=" in comment:
                key, value = comment.split("=", 1)
                metadata[key.strip()] = value.strip()
            else:
                parts = comment.split(None, 1)
                if len(parts) == 2:
                    metadata[parts[0].strip()] = parts[1].strip()
                else:
                    metadata[comment] = None
    return metadata


def combine_csvs_from_subfolders(
    root_folder,
    pattern="*.csv",
    recursive=True,
    read_kwargs=None,
    max_files=None,
    include_metadata=True,
):
    """
    Read CSV files from a folder (optionally including subfolders)
    and combine them into a single pandas DataFrame.

    Parameters
    ----------
    root_folder : str | Path
        Folder that contains CSV files directly or inside subfolders.
    pattern : str, default "*.csv"
        File name pattern to match.
    recursive : bool, default True
        If True, search all subfolders with rglob; otherwise only top-level files.
    read_kwargs : dict | None
        Optional keyword args passed to pd.read_csv (e.g., {"comment": "#"}).
    max_files : int | None
        Optional limit on number of files to read (useful for large folders).
    include_metadata : bool, default True
        If True, parse comment-block metadata and append as columns.

    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame containing all matched CSV files.
    """
    root = Path(root_folder)
    if not root.exists() or not root.is_dir():
        raise ValueError(f"Invalid folder path: {root}")

    read_kwargs = read_kwargs or {}
    csv_files = sorted(root.rglob(pattern) if recursive else root.glob(pattern))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under: {root}")

    if max_files is not None:
        if max_files <= 0:
            raise ValueError("max_files must be a positive integer")
        csv_files = csv_files[:max_files]

    frames = []
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, **read_kwargs)

        if include_metadata:
            metadata = _parse_argo_metadata(csv_file)
            for key, value in metadata.items():
                safe_key = re.sub(r"[^0-9a-zA-Z_]+", "_", key.strip().lower()).strip("_")
                meta_col = f"meta_{safe_key}" if safe_key else "meta_unknown"
                df[meta_col] = value

        frames.append(df)

    return pd.concat(frames, ignore_index=True)



In [ ]:
# Find the platform folder with the most profile CSV files and load it.
profiles_root = Path("data/raw/profiles")
platform_folders = sorted([p for p in profiles_root.iterdir() if p.is_dir()])
if not platform_folders:
    raise FileNotFoundError(f"No platform folders found under: {profiles_root}")

platform_counts = pd.Series(
    {p.name: len(list(p.glob("*.csv"))) for p in platform_folders},
    name="profile_count",
).sort_values(ascending=False)

top_10_platforms = platform_counts.head(10)
best_socal_platform = top_10_platforms.index[0]
platform_folder = profiles_root / str(best_socal_platform)

single_argo = combine_csvs_from_subfolders(
    platform_folder,
    read_kwargs={"comment": "#"},
    include_metadata=True,
)

single_argo.shape

meta_cols = [c for c in single_argo.columns if c.startswith("meta_")]
print("Top 10 platform folders by profile CSV count:")
print(top_10_platforms.to_string())
print()
print("Top platform folder:", best_socal_platform)
print("Rows:", len(single_argo))
print("Metadata columns:", len(meta_cols))
print("First metadata columns:", meta_cols[:8])
single_argo.head()

In [ ]:
# Save the top-platform dataset to CSV for reuse.
output_path = Path("data/processed/single_argo.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
single_argo.to_csv(output_path, index=False)
print("Saved", output_path, "for platform", best_socal_platform)

## Southern California Coastal Platform Search

This section uses the EasyOneArgoTSLite index file to find platform numbers near the Southern California coast, ranks them by indexed profile count, and uses the top platform to build `single_argo`.

In [ ]:
from pathlib import Path
import pandas as pd

profiles_root = Path("data/raw/profiles")
platform_folders = sorted([p for p in profiles_root.iterdir() if p.is_dir()])
if not platform_folders:
    raise FileNotFoundError(f"No platform folders found under: {profiles_root}")

# Count profile CSV files by platform folder and rank descending.
platform_counts = pd.Series(
    {p.name: len(list(p.glob("*.csv"))) for p in platform_folders},
    name="profile_count",
).sort_values(ascending=False)

top_10_platforms = platform_counts.head(10)
best_socal_platform = top_10_platforms.index[0]

print(f"Available platform folders: {len(platform_folders):,}")
print("Top 10 platform folders by profile count:")
print(top_10_platforms.to_string())
print()
print(f"Best match: platform folder {best_socal_platform} with {top_10_platforms.iloc[0]} profiles")

# Optional: inspect a sample file path from the top folder.
best_folder_files = sorted((profiles_root / str(best_socal_platform)).glob("*.csv"))
if best_folder_files:
    print("Sample file:", best_folder_files[0])

In [ ]:
# Overwrite processed output using the top platform folder.
single_argo = combine_csvs_from_subfolders(
    Path("data/raw/profiles") / str(best_socal_platform),
    read_kwargs={"comment": "#"},
    include_metadata=True,
)

output_path = Path("data/processed/single_argo.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
single_argo.to_csv(output_path, index=False)
print("Replaced", output_path, "with platform", best_socal_platform)
print("Rows:", len(single_argo))